# Temporal Gait Transformer (TGT) — Full Pipeline

**What this notebook does:**
1. Clones the repo and installs dependencies
2. Preprocesses raw insole CSV data (merge L/R, resample 60 Hz, sliding window)
3. Trains the TGT model with train/val split, LR scheduler, best-model saving
4. Evaluates on the internal held-out **test set** (confusion matrix + classification report)
5. Evaluates on the **Group 10 external validation set** (Week 3)
6. Saves all figures to `figures/`

**To use:** Runtime → Change runtime type → **T4 GPU**, then Run All.

In [ ]:
#@title 1. Setup — clone repo & install deps
import os


if not os.path.exists("sebi.py"):
    !git clone https://github.com/sebinkooooo/mech.git   
    os.chdir("mech")

%pip install -q torch torchvision torchaudio torchinfo seaborn scikit-learn tqdm matplotlib

HOME = os.getcwd()
print("Working directory:", HOME)

In [ ]:
#@title 2. Hyperparameters  (edit this cell to tune)
# ── Data / windowing ─────────────────────────────────────────────────────
WINDOW_SIZE       = 60       # timesteps per input segment  (60 = 1.0 s ≈ 1 gait cycle)
STRIDE            = 30       # step between windows (50% overlap — less redundancy)

# ── Model (Transformer) ─────────────────────────────────────────────────
D_MODEL           = 64       # embedding dimension per token
NUM_HEADS         = 4        # attention heads
NUM_LAYERS        = 2        # stacked Transformer encoder layers
DIM_FEEDFORWARD   = 256      # FFN hidden dim inside each layer
DROPOUT           = 0.3      # dropout rate (higher to combat overfitting)

# ── Training ─────────────────────────────────────────────────────────────
BATCH_SIZE        = 64
LEARNING_RATE     = 1e-4
WEIGHT_DECAY      = 1e-3     # AdamW L2 regularisation (stronger)
NUM_EPOCHS        = 120      # more epochs — stronger augmentation slows convergence
PATIENCE          = 25       # early stopping patience (epochs without val improvement)
LABEL_SMOOTHING   = 0.1      # soften targets to prevent overconfident predictions
SCHEDULER         = True     # use cosine-annealing LR scheduler

# ── Data flags ───────────────────────────────────────────────────────────
STANDARDIZE       = True     # z-score normalisation per feature (fit on TRAIN only)
NORMALIZE         = False    # min-max [0,1] per feature
WINDOW_NORM       = True     # per-window z-score (removes subject-specific baselines)
AUGMENT           = True     # training-time augmentation (noise, scaling, time mask)

# ── Reproducibility ──────────────────────────────────────────────────────
SEED              = 42

# ── Paths ────────────────────────────────────────────────────────────────
TRAIN_DATA_DIR    = os.path.join(HOME, "train_dataset")    # training CSVs
GROUP10_DATA_DIR  = os.path.join(HOME, "Group_10")         # external val
FIGURES_DIR       = os.path.join(HOME, "figures")
MODEL_SAVE_DIR    = os.path.join(HOME, "Saved_model")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [ ]:
#@title 3. Imports & seeding
import random, torch, numpy as np

from sebi import (
    FEATURE_COLUMNS, LABEL_MAPPING, CLASS_NAMES,
    load_and_window, split_by_file,
    GaitDataset, TGTModel, compute_class_weights,
    train_one_epoch, evaluate, predict_all,
    plot_training_curves, plot_confusion, plot_per_class_accuracy,
)
from torch.utils.data import DataLoader

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Week 1 — Data Preprocessing

In [ ]:
#@title 4. Load & window ALL available data (train_dataset + Group_10)
# Load main training set
feat_train, lab_train, fid_train = load_and_window(
    TRAIN_DATA_DIR, FEATURE_COLUMNS, LABEL_MAPPING, WINDOW_SIZE, STRIDE
)
print(f"train_dataset : {feat_train.shape[0]} windows from {len(np.unique(fid_train))} files")

# Load Group 10 (no Jumping class)
g10_label_map = {k: v for k, v in LABEL_MAPPING.items() if k != "Jumping"}
feat_g10, lab_g10, fid_g10 = load_and_window(
    GROUP10_DATA_DIR, FEATURE_COLUMNS, g10_label_map, WINDOW_SIZE, STRIDE
)
print(f"Group_10      : {feat_g10.shape[0]} windows from {len(np.unique(fid_g10))} files")

# Combine — offset Group 10 file IDs so they don't collide
fid_g10 = fid_g10 + fid_train.max() + 1
features = np.concatenate([feat_train, feat_g10], axis=0)
labels   = np.concatenate([lab_train, lab_g10], axis=0)
file_ids = np.concatenate([fid_train, fid_g10], axis=0)

print("=" * 50)
print(f"Total samples : {features.shape[0]}")
print(f"Unique files  : {len(np.unique(file_ids))}")
print(f"Window size   : {WINDOW_SIZE}  ({WINDOW_SIZE/60:.2f} s)")
print(f"Stride        : {STRIDE}  (overlap {1-STRIDE/WINDOW_SIZE:.0%})")
print(f"Feature shape : {features.shape}")
print(f"Classes       : {dict(zip(CLASS_NAMES, np.bincount(labels, minlength=5)))}")

In [ ]:
#@title 5. Create datasets & splits (90 / 10) — FILE-LEVEL split
# Real test is the unseen evaluation data, so we maximize training
# and keep a small val set only for early stopping / checkpointing.
train_idx, val_idx, _ = split_by_file(
    file_ids, train_ratio=0.9, val_ratio=0.1, seed=SEED
)

print(f"Files in train : {len(np.unique(file_ids[train_idx]))}")
print(f"Files in val   : {len(np.unique(file_ids[val_idx]))}")

# Fit scaler on training split only
train_ds = GaitDataset(features[train_idx], labels[train_idx],
                       standardize=STANDARDIZE, normalize=NORMALIZE,
                       window_norm=WINDOW_NORM, augment=AUGMENT)
train_scaler = train_ds.scaler

val_ds = GaitDataset(features[val_idx], labels[val_idx],
                     scaler=train_scaler, window_norm=WINDOW_NORM)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_weights = compute_class_weights(labels[train_idx], num_classes=5).to(device)
print(f"Train {len(train_ds)} | Val {len(val_ds)}")
print(f"Class weights: {class_weights}")

## Week 2 — Model Design & Training

In [ ]:
#@title 6. Build TGT model & print summary
from torchinfo import summary
import torch.nn as nn

num_features = features.shape[2]

model = TGTModel(
    num_features=num_features,
    window_size=WINDOW_SIZE,
    num_classes=5,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(device)

summary(model, input_size=(BATCH_SIZE, WINDOW_SIZE, num_features))

In [ ]:
#@title 7. Training loop (with LR scheduler, early stopping & best-model checkpointing)
import torch.optim as optim

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = None
if SCHEDULER:
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0.0
epochs_without_improvement = 0
best_model_path = os.path.join(MODEL_SAVE_DIR, "tgt_best.pth")

for epoch in range(1, NUM_EPOCHS + 1):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_loss, v_acc = evaluate(model, val_loader, criterion, device)

    train_losses.append(t_loss); train_accs.append(t_acc)
    val_losses.append(v_loss);   val_accs.append(v_acc)

    if scheduler:
        scheduler.step()

    # checkpoint best model
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        epochs_without_improvement += 1

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:3d}/{NUM_EPOCHS} │ "
          f"Train {t_loss:.4f} / {t_acc:.2f}% │ "
          f"Val {v_loss:.4f} / {v_acc:.2f}% │ "
          f"LR {lr_now:.2e}"
          + (" ★" if epochs_without_improvement == 0 else ""))

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)")
        break

print(f"\nBest val accuracy: {best_val_acc:.2f}%  →  {best_model_path}")

# save final model too
final_path = os.path.join(MODEL_SAVE_DIR, "tgt_final.pth")
torch.save(model.state_dict(), final_path)
print(f"Final model saved: {final_path}")

In [ ]:
#@title 8. Plot training curves
plot_training_curves(
    train_losses, val_losses, train_accs, val_accs,
    save_path=os.path.join(FIGURES_DIR, "train_val_curves.png"),
)
# also display inline
from IPython.display import Image, display
display(Image(filename=os.path.join(FIGURES_DIR, "train_val_curves.png")))

## Week 3 — Evaluation

### Validation set (held-out 10 % of files)
The real evaluation is on completely unseen data provided by the assignment.
This validation set is used only for early stopping and sanity checking.

In [ ]:
#@title 9. Load best model & evaluate on validation split
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)

val_loss, val_acc = evaluate(model, val_loader, criterion, device)
print("=" * 50)
print(f"Val Loss     : {val_loss:.4f}")
print(f"Val Accuracy : {val_acc:.2f}%")
print("=" * 50)

# Confusion matrix
plot_confusion(
    model, val_loader, CLASS_NAMES, device,
    save_path=os.path.join(FIGURES_DIR, "confusion_matrix_val.png"),
    title="Confusion Matrix — Validation Set (file-level split)",
)
display(Image(filename=os.path.join(FIGURES_DIR, "confusion_matrix_val.png")))

# Per-class accuracy
plot_per_class_accuracy(
    model, val_loader, CLASS_NAMES, device,
    save_path=os.path.join(FIGURES_DIR, "per_class_accuracy_val.png"),
)
display(Image(filename=os.path.join(FIGURES_DIR, "per_class_accuracy_val.png")))

### Model saved — ready for evaluation on unseen data

Group 10 data is included in the training pool above.
The saved model at `Saved_model/tgt_best.pth` will be evaluated against
the assignment's unseen test data.

In [ ]:
#@title 10. Print saved model paths
print(f"Best model : {best_model_path}")
print(f"Final model: {final_path}")
print(f"\nTrained on {len(np.unique(file_ids))} files ({features.shape[0]} windows)")
print(f"  from train_dataset/ + Group_10/")
print(f"  Classes: {CLASS_NAMES}")

In [ ]:
#@title 11. Summary: list all saved figures
print("All figures saved in:", FIGURES_DIR)
for f in sorted(os.listdir(FIGURES_DIR)):
    print(f"  • {f}")

---

## Tunable Hyperparameters (all in Cell 2)

| Parameter | Variable | Default | Description |
|-----------|----------|---------|-------------|
| Window size | `WINDOW_SIZE` | 20 | Timesteps per input segment (20 = 0.33 s at 60 Hz) |
| Stride | `STRIDE` | 5 | Step between windows; lower = more overlap = more training data |
| Embedding dim | `D_MODEL` | 128 | Dimensionality of per-token embeddings |
| Attention heads | `NUM_HEADS` | 4 | Multi-head attention heads |
| Encoder layers | `NUM_LAYERS` | 4 | Stacked Transformer encoder blocks |
| FFN hidden dim | `DIM_FEEDFORWARD` | 512 | Feed-forward hidden size inside each layer |
| Dropout | `DROPOUT` | 0.1 | Dropout rate in Transformer |
| Batch size | `BATCH_SIZE` | 64 | Mini-batch size |
| Learning rate | `LEARNING_RATE` | 3e-4 | Initial learning rate |
| Weight decay | `WEIGHT_DECAY` | 1e-4 | L2 regularisation (AdamW) |
| Epochs | `NUM_EPOCHS` | 60 | Total training epochs |
| LR scheduler | `SCHEDULER` | True | Cosine annealing LR decay |
| Standardize | `STANDARDIZE` | False | Per-feature z-score normalisation |
| Normalize | `NORMALIZE` | False | Per-feature min-max to [0, 1] |
| Seed | `SEED` | 42 | Random seed for reproducibility |
| Feature set | `FEATURE_COLUMNS` | (in sebi.py) | Which sensor channels to use |
| Label map | `LABEL_MAPPING` | (in sebi.py) | Activity name → class index |